In [ ]:
import pandas as pd
import numpy as np
import glob


def _to_bool_flag(x):
    """Convert typical truthy/falsey strings to bool."""
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"1", "True", "true", "yes"}:
            return True
        if s in {"0", "False", "false", "no"}:
            return False
    return bool(x)

def _parse_float_or_none(x):
    try:
        return float(str(x).strip())
    except Exception:
        return None

def _answer_value_correct(gt_val, pred_val, rel_tol=1e-3):
    """
    gt_val, pred_val: values from answer_value columns.
    rel_tol = 0.001 => 0.1% relative tolerance.
    """
    gt_str = str(gt_val).strip()
    pred_str = str(pred_val).strip()
    
    # If either is 'is_blank', treat as categorical
    if gt_str.lower() == "is_blank" or pred_str.lower() == "is_blank":
        return gt_str.lower() == pred_str.lower()
    
    gt_num = _parse_float_or_none(gt_val)
    pred_num = _parse_float_or_none(pred_val)
    
    # If both numeric, use relative tolerance
    if gt_num is not None and pred_num is not None:
        if gt_num == 0:
            return abs(pred_num - gt_num) <= rel_tol  # small absolute tolerance around 0
        rel_err = abs(pred_num - gt_num) / max(abs(gt_num), 1e-12)
        return rel_err <= rel_tol
    
    # Otherwise, fall back to normalized string match
    return gt_str.lower() == pred_str.lower()

def _ref_id_jaccard(gt_ref, pred_ref):
    """
    Jaccard overlap between sets of ref_ids.
    Strings may contain semicolon-separated IDs, or 'is_blank'.
    Case-insensitive.
    """
    def to_set(s):
        if s is None:
            return set()
        s = str(s).strip()
        if not s or s.lower() == "is_blank":
            return set()
        parts = [p.strip().lower() for p in s.split(";") if p.strip()]
        return set(parts)
    
    gt_set = to_set(gt_ref)
    pred_set = to_set(pred_ref)
    
    if not gt_set and not pred_set:
        return 1.0
    union = gt_set | pred_set
    if not union:
        return 0.0
    inter = gt_set & pred_set
    return len(inter) / len(union)


def compute_wattbot_score(
    train_qa_path="train_QA.csv",
    preds_path="train_solutions_qwen.csv",
    id_col="id",
    gt_answer_col="answer_value",
    gt_ref_col="ref_id",
    gt_is_na_col="is_NA",
    pred_answer_col="answer_value",
    pred_ref_col="ref_id",
    pred_is_na_col=None,
    n_examples=10,
    verbose=True,                 # NEW
):
    gt = pd.read_csv(train_qa_path)
    preds = pd.read_csv(preds_path)

    merged = gt.merge(preds, on=id_col, suffixes=("_gt", "_pred"))
    if merged.empty:
        raise ValueError("No overlapping ids between ground truth and predictions.")

    # ----- ground truth NA flags -----
    if gt_is_na_col is not None and gt_is_na_col in merged.columns:
        gt_is_na_series = merged[gt_is_na_col].map(_to_bool_flag)
    elif gt_is_na_col is not None and gt_is_na_col.lower() == "is_blank":
        gt_is_na_series = merged[f"{gt_answer_col}_gt"].astype(str).str.lower().eq("is_blank")
        merged["gt_is_blank_flag"] = gt_is_na_series
    else:
        if "is_NA" in merged.columns:
            gt_is_na_series = merged["is_NA"].map(_to_bool_flag)
        elif "is_blank" in merged.columns:
            gt_is_na_series = merged["is_blank"].map(_to_bool_flag)
        else:
            gt_is_na_series = merged[f"{gt_answer_col}_gt"].astype(str).str.lower().eq("is_blank")
            merged["gt_is_blank_flag"] = gt_is_na_series

    # ----- prediction NA flags -----
    if pred_is_na_col is not None and pred_is_na_col in merged.columns:
        pred_is_na_series = merged[pred_is_na_col].map(_to_bool_flag)
    elif pred_is_na_col is not None and pred_is_na_col.lower() == "is_blank":
        pred_is_na_series = merged[f"{pred_answer_col}_pred"].astype(str).str.lower().eq("is_blank")
        merged["pred_is_blank_flag"] = pred_is_na_series
    else:
        if "is_NA" in merged.columns:
            pred_is_na_series = merged["is_NA"].map(_to_bool_flag)
        elif "is_blank" in merged.columns:
            pred_is_na_series = merged["is_blank"].map(_to_bool_flag)
        else:
            pred_is_na_series = merged[f"{pred_answer_col}_pred"].astype(str).str.lower().eq("is_blank")
            merged["pred_is_blank_flag"] = pred_is_na_series

    ans_scores, ref_scores, na_scores = [], [], []

    # IMPORTANT: don't use merged.iterrows() index to index series; use enumerate
    for i, (_, row) in enumerate(merged.iterrows()):
        gt_ans   = row[f"{gt_answer_col}_gt"]
        pred_ans = row[f"{pred_answer_col}_pred"]
        gt_ref   = row[f"{gt_ref_col}_gt"]
        pred_ref = row[f"{pred_ref_col}_pred"]

        gt_is_na = bool(gt_is_na_series.iloc[i])
        pred_is_na = bool(pred_is_na_series.iloc[i])

        ans_correct = _answer_value_correct(gt_ans, pred_ans)
        ans_scores.append(float(ans_correct))

        ref_j = _ref_id_jaccard(gt_ref, pred_ref)
        ref_scores.append(ref_j)

        if gt_is_na:
            na_scores.append(1.0 if pred_is_na else 0.0)
        else:
            na_scores.append(np.nan)

    merged["answer_score"] = ans_scores
    merged["ref_id_score"] = ref_scores
    merged["is_NA_score"]  = na_scores  # NaN for answerable rows

    na_recall = merged["is_NA_score"].mean()  # ignores NaN
    na_recall_val = 0.0 if pd.isna(na_recall) else float(na_recall)

    mean_answer = float(merged["answer_score"].mean())
    mean_ref    = float(merged["ref_id_score"].mean())

    overall_score = 0.75 * mean_answer + 0.15 * mean_ref + 0.10 * na_recall_val

    merged["wattbot_score"] = overall_score  # broadcast for convenience

    if verbose:
        print(f"Rows compared: {len(merged)}")
        print(f"Mean answer_value score: {mean_answer:.4f}")
        print(f"Mean ref_id score:       {mean_ref:.4f}")
        print(f"NA recall (GT NA only):  {na_recall_val:.4f}")
        print(f"Overall WattBot score:   {overall_score:.4f}")

        incorrect = merged[
            (merged["answer_score"] < 1.0)
            | (merged["ref_id_score"] < 1.0)
            | (merged["is_NA_score"] < 1.0)
        ]
        if not incorrect.empty and n_examples > 0:
            print(f"\nExamples (up to {n_examples}):")
            for _, r in incorrect.head(n_examples).iterrows():
                print("-" * 80)
                print(f"id: {r[id_col]}")
                print(f"GT answer_value:   {r[f'{gt_answer_col}_gt']}")
                print(f"Pred answer_value: {r[f'{pred_answer_col}_pred']}")
                print(f"GT ref_id:         {r[f'{gt_ref_col}_gt']}")
                print(f"Pred ref_id:       {r[f'{pred_ref_col}_pred']}")
            print("-" * 80)

    # NEW: return score + components cleanly
    return {
        "merged": merged,
        "overall_score": overall_score,
        "mean_answer": mean_answer,
        "mean_ref": mean_ref,
        "na_recall": na_recall_val,
    }


In [8]:
# results = ["train_solutions_Qwen__Qwen2.5-7B-Instruct.csv", "train_solutions_Qwen__Qwen2.5-32B-Instruct.csv", "train_solutions_Qwen__Qwen2.5-72B-Instruct.csv"]
# for result_file in results:
#     results_df = compute_wattbot_score(
#         train_qa_path="./data/train_QA.csv",
#         preds_path=f"./data/{result_file}",
#         gt_is_na_col="is_blank",   # or "is_blank" / None depending on how you mark NAs
#         n_examples=0,
#     )

In [4]:
glob.glob("data/train_solutions*_run_metrics.json")
glob.glob("data/train_solutions*.csv")

['data/train_solutions_NVIDIA2_Qwen__Qwen2.5-32B-Instruct.csv',
 'data/train_solutions_NVIDIA2_Qwen__Qwen2.5-3B-Instruct.csv',
 'data/train_solutions_NVIDIA2_Qwen__Qwen2.5-72B-Instruct.csv',
 'data/train_solutions_NVIDIA2_Qwen__Qwen2.5-7B-Instruct.csv',
 'data/train_solutions_Qwen__Qwen2.5-32B-Instruct.csv',
 'data/train_solutions_Qwen__Qwen2.5-72B-Instruct.csv',
 'data/train_solutions_Qwen__Qwen2.5-7B-Instruct.csv']

In [5]:
# !pip install matplotlib

In [6]:
import os, re, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------- FIND FILES (robust) ----------
# Option 1: current folder only
csv_paths = glob.glob("data/train_solutions*.csv")

# Option 2: search recursively under current folder (uncomment if needed)
# csv_paths = glob.glob("**/train_solutions*.csv", recursive=True)

print(f"Found {len(csv_paths)} CSV files")
for p in csv_paths[:10]:
    print("  ", p)

if len(csv_paths) == 0:
    raise FileNotFoundError(
        "No files matched 'train_solutions*.csv'.\n"
        "Fix: run this in the directory containing the CSVs, or use recursive glob."
    )

# --------- helpers ----------
def system_from_filename(path: str) -> str:
    bn = os.path.basename(path)
    if "NVIDIA1" in bn: return "NVIDIA1"
    if "NVIDIA2" in bn: return "NVIDIA1"
    return "GB10"

def model_id_from_csv_filename(path: str) -> str:
    bn = os.path.basename(path)
    # expects "...__Qwen2.5-7B-Instruct.csv"
    if "__" not in bn:
        # fallback: just store filename
        return bn.replace(".csv", "")
    after = bn.split("__", 1)[1].replace(".csv", "")
    return "Qwen/" + after

def model_b_from_model_id(model_id: str) -> float:
    m = re.search(r"-(\d+)B-", model_id)
    return float(m.group(1)) if m else np.nan

def metrics_json_for_csv(csv_path: str) -> str | None:
    j = csv_path.replace(".csv", "_run_metrics.json")
    return j if os.path.exists(j) else None

# --------- your plotting fn ----------
def plot_lines(df: pd.DataFrame, ycol: str, ylabel: str, title: str, out_path: str):
    x_offset = {"GB10": -0.25, "NVIDIA1": 0.00, "NVIDIA2": 0.25}

    plt.figure()
    for sys, g in df.groupby("system"):
        g = g.sort_values("model_b")
        xs = g["model_b"].to_numpy() + x_offset.get(sys, 0.0)
        plt.plot(xs, g[ycol], marker="o", linewidth=2, label=sys, zorder=3)
        plt.scatter(xs, g[ycol], s=80, zorder=4)

    plt.xlabel("Model size (B parameters)")
    plt.ylabel(ylabel)
    plt.title(title)
    base_ticks = sorted([x for x in df["model_b"].unique() if not np.isnan(x)])
    plt.xticks(base_ticks)
    plt.legend()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

# --------- BUILD SUMMARY ----------
train_qa_path = "data/train_QA.csv"  # update to full path if needed

rows = []
for csv_path in csv_paths:
    system = system_from_filename(csv_path)
    model_id = model_id_from_csv_filename(csv_path)
    model_b = model_b_from_model_id(model_id)

    # score (silent)
    score_out = compute_wattbot_score(
        train_qa_path=train_qa_path,
        preds_path=csv_path,
        gt_is_na_col="is_NA",
        pred_is_na_col="is_blank",
        n_examples=0,
        verbose=False,
    )

    row = {
        "system": system,
        "model_id": model_id,
        "model_b": model_b,
        "csv_path": csv_path,
        "overall_wattbot_score": score_out["overall_score"],
    }

    # latency (if json exists)
    jpath = metrics_json_for_csv(csv_path)
    if jpath:
        with open(jpath, "r") as f:
            d = json.load(f)
        t = d.get("timings", {})
        num_q = t.get("num_questions", None)
        gen_s = t.get("timing_generation_s", None)
        emb_s = t.get("timing_embedding_s", 0.0)

        row["gen_s_per_q"] = (gen_s / num_q) if (gen_s is not None and num_q) else np.nan
        row["end_to_end_s_per_q"] = ((gen_s + emb_s) / num_q) if (gen_s is not None and num_q) else np.nan
    else:
        row["gen_s_per_q"] = np.nan
        row["end_to_end_s_per_q"] = np.nan

    rows.append(row)

summary_df = pd.DataFrame(rows)

# sanity check
print("\nSummary columns:", list(summary_df.columns))
print(summary_df[["system","model_b","overall_wattbot_score","end_to_end_s_per_q","gen_s_per_q"]])

summary_df = summary_df.sort_values(["system", "model_b"])

# --------- PLOTS ----------
plot_lines(
    summary_df,
    ycol="overall_wattbot_score",
    ylabel="Overall WattBot score",
    title="Model size vs Overall WattBot score",
    out_path="model_size_vs_wattbot_score.png",
)

plot_lines(
    summary_df,
    ycol="end_to_end_s_per_q",
    ylabel="Seconds per question (embed + generation)",
    title="Model size vs latency (end-to-end)",
    out_path="model_size_vs_latency_end_to_end.png",
)


Found 7 CSV files
   data/train_solutions_NVIDIA2_Qwen__Qwen2.5-32B-Instruct.csv
   data/train_solutions_NVIDIA2_Qwen__Qwen2.5-3B-Instruct.csv
   data/train_solutions_NVIDIA2_Qwen__Qwen2.5-72B-Instruct.csv
   data/train_solutions_NVIDIA2_Qwen__Qwen2.5-7B-Instruct.csv
   data/train_solutions_Qwen__Qwen2.5-32B-Instruct.csv
   data/train_solutions_Qwen__Qwen2.5-72B-Instruct.csv
   data/train_solutions_Qwen__Qwen2.5-7B-Instruct.csv


NameError: name '_to_bool_flag' is not defined